# **Manual trading challenge: “Invest & Expand”**

You are expanding your outpost into a true market making firm with a budget of `50 000` XIRECs. You need to allocate this budget across three pillars:

- **Research**
- **Scale**
- **Speed**

You choose percentages for each pillar between 0–100%. Total allocation cannot exceed 100%. Your final PnL (Profit and Loss) score is:

<aside>
ℹ️

PnL = (Research × Scale × Speed) − Budget_Used

</aside>

### **The pillars**

**Research** determines how strong your trading edge is. It grows **logarithmically** from `0` (for `0` invested) to `200 000`  (for `100` invested). The exact formula is `research(x) = 200_000 * np.log(1 + x) / np.log(1 + 100)`. Here, `np.log` is a python function from NumPy package for natural logarithm.

**Scale** determines how broadly you deploy your strategy across markets. It grows **linearly** from `0` (for `0` invested) to `7` (for `100` invested).

**Speed** determines how often you win the trades you target. It is **rank-based** across all players:

- Highest speed investment receives a `0.9` multiplier.
- Lowest receives `0.1`.
- Everyone in between is scaled linearly by rank, equal investments share the same rank.
- For example, if people invested `70, 70, 70, 50, 40, 40, 30`, they get the following ranks: `1, 1, 1, 4, 5, 5, 7`. First three players get `0.9` for hit rate multiplier, last player gets `0.1`, and everybody in between gets linearly scaled between top and bottom rank. Another example, if you have three players investing `95, 20, 10`, their ranks are `1, 2, 3`, and their hit rates are `0.9, 0.5, 0.1`.

Your Research, Scale, and Speed outcomes are multiplied together to form your gross PnL, after which the used part of your budget is deducted.

Every decision you make reflects a real trade-off faced by modern market makers: capital is finite, competition is relentless, and edge alone is never enough. Good luck!

In [ ]:
from pathlib import Path
import sys

import numpy as np

repo_root = Path.cwd()
if not (repo_root / "datasets" / "round2" / "speed_optimizer.py").exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root / "datasets" / "round2"))

from speed_optimizer import (
    DiscreteSpeedDistribution,
    expected_speed_from_distribution,
    optimize_parameters_from_distribution,
)


In [ ]:
# Baseline 20,000-player model for the speed market.
values = np.arange(0, 101)

# 50.0% random bettors: uniform over the full c range.
random_probs = np.ones(101) / 101

# 33.5% intuitive bettors: round numbers and the middle compromise region.
intuitive_probs = np.zeros(101)
intuitive_probs[[0, 10, 20, 30, 40, 50, 60, 70, 80]] = [
    0.05, 0.05, 0.10, 0.20, 0.25, 0.20, 0.10, 0.04, 0.01,
]

# 7.5% GTO bettors: spread across focal fixed points rather than one spike.
gto_probs = np.zeros(101)
gto_probs[[0, 20, 40, 60, 80]] = [
    0.05, 0.15, 0.40, 0.25, 0.15,
]

# 9.0% perfect-response bettors: concentrated at the self-consistent c*=40.
perfect_probs = np.zeros(101)
perfect_probs[40] = 1.0

probs = (
    0.50 * random_probs
    + 0.335 * intuitive_probs
    + 0.075 * gto_probs
    + 0.09 * perfect_probs
)

speed_distribution = DiscreteSpeedDistribution.from_pairs(
    values=values,
    probabilities=probs,
)

{
    "probs_sum": float(probs.sum()),
    "p(c=0)": float(probs[0]),
    "p(c=20)": float(probs[20]),
    "p(c=30)": float(probs[30]),
    "p(c=40)": float(probs[40]),
    "p(c=50)": float(probs[50]),
    "p(c=60)": float(probs[60]),
    "p(c=90)": float(probs[90]),
    "p(c=100)": float(probs[100]),
}


In [ ]:
# Model the full field size.
n_opponents = 19_999

best = optimize_parameters_from_distribution(
    distribution=speed_distribution,
    n_opponents=n_opponents,
    step=1.0,
)

best


In [ ]:
def transfer_mass(probabilities, transfers):
    shifted = probabilities.copy()
    for source, target, mass in transfers:
        if shifted[source] < mass:
            raise ValueError(f"cannot move {mass} from {source}")
        shifted[source] -= mass
        shifted[target] += mass
    return shifted


def build_mixture(intuitive_component, gto_component, perfect_c=40):
    perfect_component = np.zeros(101)
    perfect_component[perfect_c] = 1.0
    return DiscreteSpeedDistribution.from_pairs(
        values=values,
        probabilities=(
            0.50 * random_probs
            + 0.335 * intuitive_component
            + 0.075 * gto_component
            + 0.09 * perfect_component
        ),
    )


higher_speed_distribution = build_mixture(
    intuitive_component=transfer_mass(
        intuitive_probs,
        [(30, 50, 0.05), (40, 60, 0.05)],
    ),
    gto_component=transfer_mass(
        gto_probs,
        [(40, 60, 0.05)],
    ),
)

lower_speed_distribution = build_mixture(
    intuitive_component=transfer_mass(
        intuitive_probs,
        [(50, 30, 0.05), (60, 20, 0.05)],
    ),
    gto_component=transfer_mass(
        gto_probs,
        [(60, 20, 0.05)],
    ),
)

{
    "expected_speed_at_40": expected_speed_from_distribution(
        40,
        speed_distribution,
        n_opponents=n_opponents,
    ),
    "higher_speed_best": optimize_parameters_from_distribution(
        higher_speed_distribution,
        n_opponents=n_opponents,
        step=1.0,
    ),
    "lower_speed_best": optimize_parameters_from_distribution(
        lower_speed_distribution,
        n_opponents=n_opponents,
        step=1.0,
    ),
}
